In [ ]:
#Code with out Langchain framwework


import os
from google.colab import files, userdata
import google.genai as genai

# For PDF and DOCX text extraction, which are already installed
import pypdf
import docx2txt

# Step 3 - API key setup (from original notebook)
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# Step 4 - LLM (replacing Langchain's ChatGoogleGenerativeAI)
llm = genai.GenerativeModel("gemini-2.5-flash")

# Step 5 - Create a prompt (replacing Langchain's PromptTemplate)
def create_prompt(job_description, resume_text):
    template = """
    You are going to work as AI Resume Screener

    Job Description:
    {job_description}

    Resume Text:
    {resume_text}

    Give a simple analysis:
    - FIT Score(0-100)
    - Top 5 matching skills
    - Missing Important Skills
    - One-line-Verdict
    """
    return template.format(job_description=job_description, resume_text=resume_text)

# Function to extract text from various document types (replacing Langchain loaders)
def extract_text_from_file(file_path):
    text = ""
    if file_path.lower().endswith(".pdf"):
        reader = pypdf.PdfReader(file_path)
        for page in reader.pages:
            text += page.extract_text() or ""
    elif file_path.lower().endswith(".docx"):
        text = docx2txt.process(file_path)
    elif file_path.lower().endswith(".txt"):
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
    else:
        raise ValueError("Unsupported file format. Please upload PDF, DOCX, or TXT.")
    return text

# Step 6 - Upload your resume and extract text out of it
print("Please Upload your resume (PDF/DOCX/TXT):")
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file uploaded")

resume_path = list(uploaded.keys())[0]
resume_text = extract_text_from_file(resume_path)

if not resume_text:
    raise ValueError("Could not extract text from uploaded file")

# Step 7 : Run analysis and define job description
job_description = """ We are hiring for a Lead Python Developer with experience in Python, AI, SQL, Cloud(Aws/GCP),and Data Analysis"""

# Generate prompt and call LLM (replacing Langchain's LLMChain.run)
prompt = create_prompt(job_description=job_description, resume_text=resume_text)
response = llm.generate_content(prompt)

print("=================Result Analysis==============")
print(response.text)
